<a href="https://colab.research.google.com/github/KishanVyas308/CatGPT/blob/main/cat_gpt_v1.1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## 🐾 Setup & Data Gathering

*Meow! Before we start hunting for mice, we must gather our cat-pack libraries! We are installing Hugging Face's `datasets` library to fetch our delicious chat feeds. Think of this as getting a fresh can of premium tuna!*

In [4]:
!pip install -U -q datasets

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 559.1/559.1 kB 34.4 MB/s eta 0:00:00


In [5]:
from datasets import load_dataset

dataset = load_dataset("elricwan/dailydialog")

print(dataset)

README.md:   0%|          | 0.00/348 [00:00<?, ?B/s]

data/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 4.06MB            

data/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/13118 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['conversation'],
        num_rows: 13118
    })
})


### 🐱 Inspection Time!

*A good cat always sniffs its food before eating. Here, we are inspecting our dataset columns and first entries to see exactly what we're working with. Let's peek into the bowl!*

In [7]:
print(dataset)

DatasetDict({
    train: Dataset({
        features: ['conversation'],
        num_rows: 13118
    })
})


In [8]:
print(dataset["train"].column_names)

['conversation']


In [9]:
print(dataset["train"][0])

{'conversation': ['Person A: The kitchen stinks .', "Person B: I'll throw out the garbage ."]}


In [10]:
sample = dataset["train"][0]
print(sample)

{'conversation': ['Person A: The kitchen stinks .', "Person B: I'll throw out the garbage ."]}


## Step - 4

- Convert Person 2 to Cat




### 😹 Turning Humans Into Cats

*Humans are boring, so we are converting 'Person B' into 'Cat' while keeping 'Person A' as 'Human'. We'll use parallel processing to speed this up, because a lazy cat wants its nap as quickly as possible!*

In [13]:
import os
import torch
from concurrent.futures import ThreadPoolExecutor

# -------------------------
# Check available resources
# -------------------------
print("CPU cores:", os.cpu_count())
print("GPU available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))


# -------------------------
# Load conversations
# -------------------------
conversations = dataset["train"]["conversation"]

print("Total conversations:", len(conversations))


# -------------------------
# Convert one conversation
# -------------------------
def process_conversation(conversation):
    lines = []

    for turn in conversation:
        turn = turn.replace("Person A:", "Human:")
        turn = turn.replace("Person B:", "Cat:")
        lines.append(turn)

    return "\n".join(lines) + "\n\n"


# -------------------------
# Parallel CPU processing
# -------------------------
workers = min(8, os.cpu_count() or 1)

with ThreadPoolExecutor(max_workers=workers) as executor:
    processed = list(
        executor.map(process_conversation, conversations)
    )


# -------------------------
# Combine everything
# -------------------------
cat_text = "".join(processed)


# -------------------------
# Save dataset
# -------------------------
with open("cat_data.txt", "w", encoding="utf-8") as f:
    f.write(cat_text)


# -------------------------
# Statistics
# -------------------------
print("\n===== CatGPT Dataset =====")
print("Conversations :", len(conversations))
print("Characters     :", len(cat_text))
print("File size      :", os.path.getsize("cat_data.txt") / 1024, "KB")

print("\n===== Sample =====")
print(cat_text[:2000])

CPU cores: 2
GPU available: True
GPU: Tesla T4
Total conversations: 13118

===== CatGPT Dataset =====
Conversations : 13118
Characters     : 6926506
File size      : 6764.8291015625 KB

===== Sample =====
Human: The kitchen stinks .
Cat: I'll throw out the garbage .

Human: So Dick , how about getting some coffee for tonight ?
Cat: Coffee ? I don ' t honestly like that kind of stuff .
Human: Come on , you can at least try a little , besides your cigarette .
Cat: What ' s wrong with that ? Cigarette is the thing I go crazy for .
Human: Not for me , Dick .

Human: Are things still going badly with your houseguest ?
Cat: Getting worse . Now he ' s eating me out of house and home . I ' Ve tried talking to him but it all goes in one ear and out the other . He makes himself at home , which is fine . But what really gets me is that yesterday he walked into the living room in the raw and I had company over ! That was the last straw .
Human: Leo , I really think you ' re beating around the bush

## Step 5: build the tokenizer from scratch. 🐱

### ✂️ Shredding the Text (Custom Tokenizer)

*Time to scratch this text file into tiny bits! Instead of yarn, we are slicing the text down to unique character tokens so our model can learn the alphabet of meowing. We map every character to a specific number—it's like tracking paw prints!*

In [14]:
import torch

# ============================================
# STEP 5: CHARACTER-LEVEL TOKENIZER
# ============================================

# Load our CatGPT dataset
with open("cat_data.txt", "r", encoding="utf-8") as f:
    text = f.read()

print("Dataset characters:", len(text))


# --------------------------------------------
# 1. Build vocabulary
# --------------------------------------------

# Get every unique character in our dataset
chars = sorted(list(set(text)))

vocab_size = len(chars)

print("Vocabulary size:", vocab_size)
print("Vocabulary:")
print(chars)


# --------------------------------------------
# 2. Create character ↔ integer mappings
# --------------------------------------------

# character -> integer
stoi = {ch: i for i, ch in enumerate(chars)}

# integer -> character
itos = {i: ch for i, ch in enumerate(chars)}


# --------------------------------------------
# 3. Encoder
# --------------------------------------------

def encode(text):
    return [stoi[c] for c in text]


# --------------------------------------------
# 4. Decoder
# --------------------------------------------

def decode(tokens):
    return "".join(itos[i] for i in tokens)


# --------------------------------------------
# 5. Test tokenizer
# --------------------------------------------

sample = "Human: Hello"

encoded = encode(sample)
decoded = decode(encoded)

print("\nOriginal:")
print(sample)

print("\nEncoded:")
print(encoded)

print("\nDecoded:")
print(decoded)


# --------------------------------------------
# 6. Encode entire dataset
# --------------------------------------------

data = torch.tensor(
    encode(text),
    dtype=torch.long
)

print("\nEncoded dataset shape:", data.shape)
print("First 100 tokens:")
print(data[:100])

Dataset characters: 6926506
Vocabulary size: 99
Vocabulary:
['\n', ' ', '!', '"', '#', '$', '%', '&', "'", '(', ')', '*', '+', ',', '-', '.', '/', '0', '1', '2', '3', '4', '5', '6', '7', '8', '9', ':', ';', '=', '?', '@', 'A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'J', 'K', 'L', 'M', 'N', 'O', 'P', 'Q', 'R', 'S', 'T', 'U', 'V', 'W', 'X', 'Y', 'Z', '\\', '_', 'a', 'b', 'c', 'd', 'e', 'f', 'g', 'h', 'i', 'j', 'k', 'l', 'm', 'n', 'o', 'p', 'q', 'r', 's', 't', 'u', 'v', 'w', 'x', 'y', 'z', '~', '\x7f', '£', '¥', '°', '–', '—', '‘', '“', '”', '′', '、', '。']

Original:
Human: Hello

Encoded:
[39, 80, 72, 60, 73, 27, 1, 39, 64, 71, 71, 74]

Decoded:
Human: Hello

Encoded dataset shape: torch.Size([6926506])
First 100 tokens:
tensor([39, 80, 72, 60, 73, 27,  1, 51, 67, 64,  1, 70, 68, 79, 62, 67, 64, 73,
         1, 78, 79, 68, 73, 70, 78,  1, 15,  0, 34, 60, 79, 27,  1, 40,  8, 71,
        71,  1, 79, 67, 77, 74, 82,  1, 74, 80, 79,  1, 79, 67, 64,  1, 66, 60,
        77, 61, 60, 66, 64,  1

## 6: prepare training batches

### 6.1 Train/validation split

In [16]:
# 90% for training
# 10% for validation

n = int(0.9 * len(data))

train_data = data[:n]
val_data = data[n:]

print("Total tokens :", len(data))
print("Train tokens :", len(train_data))
print("Val tokens   :", len(val_data))

Total tokens : 6926506
Train tokens : 6233855
Val tokens   : 692651


In [17]:
# Use GPU if available
device = "cuda" if torch.cuda.is_available() else "cpu"

print("Using:", device)

if device == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))

Using: cuda
GPU: Tesla T4


In [18]:
# ============================================
# BATCH GENERATOR
# ============================================

block_size = 128
batch_size = 64


def get_batch(split):

    # Select dataset
    dataset = train_data if split == "train" else val_data

    # Random starting positions
    ix = torch.randint(
        len(dataset) - block_size,
        (batch_size,)
    )

    # Input sequences
    x = torch.stack([
        dataset[i:i + block_size]
        for i in ix
    ])

    # Target sequences
    y = torch.stack([
        dataset[i + 1:i + block_size + 1]
        for i in ix
    ])

    # Move to GPU
    x = x.to(device)
    y = y.to(device)

    return x, y

In [19]:
xb, yb = get_batch("train")

print("Input shape :", xb.shape)
print("Target shape:", yb.shape)

Input shape : torch.Size([64, 128])
Target shape: torch.Size([64, 128])


In [20]:
print("Input:")
print(decode(xb[0].tolist()))

print("\nTarget:")
print(decode(yb[0].tolist()))

Input:
 series of animals , called Chinese zodiac .
Human: Good . So I will take three series and five bamboo ones .
Cat: OK , I will w

Target:
series of animals , called Chinese zodiac .
Human: Good . So I will take three series and five bamboo ones .
Cat: OK , I will wr


In [21]:
print(xb.device)
print(xb.shape)
print(yb.shape)

cuda:0
torch.Size([64, 128])
torch.Size([64, 128])


## Self-Attention

### 🧠 Causal Self-Attention: Focus Like a Hunter

*How does a cat focus on a moving bug while ignoring the rest of the room? Attention! We build a causal attention block that masks out future tokens, ensuring our CatGPT model only predicts what comes next, never looking ahead at the hidden laser pointer.*

In [22]:
import torch
import torch.nn as nn
import torch.nn.functional as F

# Reproducibility
torch.manual_seed(42)

# --------------------------------------------
# Tiny example
# --------------------------------------------

B = 1       # batch size
T = 8       # sequence length
C = 32      # embedding dimension

# Fake token embeddings
x = torch.randn(B, T, C)

print("Input shape:", x.shape)

Input shape: torch.Size([1, 8, 32])


In [23]:
# --------------------------------------------
# Query, Key, Value
# --------------------------------------------

head_size = 32

key = nn.Linear(C, head_size, bias=False)
query = nn.Linear(C, head_size, bias=False)
value = nn.Linear(C, head_size, bias=False)

k = key(x)
q = query(x)
v = value(x)

print("Key shape   :", k.shape)
print("Query shape :", q.shape)
print("Value shape :", v.shape)

Key shape   : torch.Size([1, 8, 32])
Query shape : torch.Size([1, 8, 32])
Value shape : torch.Size([1, 8, 32])


In [24]:
# --------------------------------------------
# Attention scores
# --------------------------------------------

scores = q @ k.transpose(-2, -1)

print("Scores shape:", scores.shape)

Scores shape: torch.Size([1, 8, 8])


In [25]:
scores = scores / (head_size ** 0.5)

In [26]:
# --------------------------------------------
# Causal mask
# --------------------------------------------

tril = torch.tril(torch.ones(T, T))

print(tril)

tensor([[1., 0., 0., 0., 0., 0., 0., 0.],
        [1., 1., 0., 0., 0., 0., 0., 0.],
        [1., 1., 1., 0., 0., 0., 0., 0.],
        [1., 1., 1., 1., 0., 0., 0., 0.],
        [1., 1., 1., 1., 1., 0., 0., 0.],
        [1., 1., 1., 1., 1., 1., 0., 0.],
        [1., 1., 1., 1., 1., 1., 1., 0.],
        [1., 1., 1., 1., 1., 1., 1., 1.]])


In [27]:
scores = scores.masked_fill(
    tril == 0,
    float("-inf")
)

In [28]:
attention = F.softmax(scores, dim=-1)

print(attention)

tensor([[[1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
         [0.5759, 0.4241, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
         [0.1996, 0.3720, 0.4283, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
         [0.2081, 0.3659, 0.1977, 0.2284, 0.0000, 0.0000, 0.0000, 0.0000],
         [0.1483, 0.1859, 0.2971, 0.1651, 0.2037, 0.0000, 0.0000, 0.0000],
         [0.2422, 0.2291, 0.1616, 0.1383, 0.0877, 0.1411, 0.0000, 0.0000],
         [0.1819, 0.1538, 0.1976, 0.1236, 0.1581, 0.0836, 0.1014, 0.0000],
         [0.1756, 0.0980, 0.1230, 0.1224, 0.1237, 0.1020, 0.0999, 0.1554]]],
       grad_fn=<SoftmaxBackward0>)


In [29]:
out = attention @ v

print("Output shape:", out.shape)

Output shape: torch.Size([1, 8, 32])


In [30]:
import torch
import torch.nn as nn
import torch.nn.functional as F


class CausalSelfAttention(nn.Module):

    def __init__(self, n_embd, n_head, dropout=0.1):
        super().__init__()

        assert n_embd % n_head == 0

        self.n_head = n_head
        self.head_dim = n_embd // n_head

        # Create Query, Key and Value together
        self.qkv = nn.Linear(
            n_embd,
            3 * n_embd
        )

        # Final projection
        self.proj = nn.Linear(
            n_embd,
            n_embd
        )

        self.dropout = dropout

    def forward(self, x):

        # x shape:
        # [batch_size, sequence_length, embedding_size]

        B, T, C = x.shape

        # --------------------------------
        # Create Q, K, V
        # --------------------------------

        q, k, v = self.qkv(x).chunk(3, dim=-1)

        # --------------------------------
        # Split into attention heads
        # --------------------------------

        q = q.view(
            B, T, self.n_head, self.head_dim
        ).transpose(1, 2)

        k = k.view(
            B, T, self.n_head, self.head_dim
        ).transpose(1, 2)

        v = v.view(
            B, T, self.n_head, self.head_dim
        ).transpose(1, 2)

        # --------------------------------
        # Causal Self-Attention
        # --------------------------------

        y = F.scaled_dot_product_attention(
            q,
            k,
            v,
            dropout_p=self.dropout if self.training else 0.0,
            is_causal=True
        )

        # --------------------------------
        # Combine attention heads
        # --------------------------------

        y = y.transpose(1, 2).contiguous()

        y = y.view(
            B,
            T,
            C
        )

        # --------------------------------
        # Output projection
        # --------------------------------

        y = self.proj(y)

        return y

In [31]:
# CatGPT v1 model dimensions

n_embd = 128
n_head = 4

attention = CausalSelfAttention(
    n_embd=n_embd,
    n_head=n_head
).to(device)


# Create fake embeddings
x = torch.randn(
    batch_size,
    block_size,
    n_embd,
    device=device
)


# Run attention
y = attention(x)


print("Device :", x.device)
print("Input  :", x.shape)
print("Output :", y.shape)

Device : cuda:0
Input  : torch.Size([64, 128, 128])
Output : torch.Size([64, 128, 128])


In [32]:
# ============================================
# STEP 8: FEED-FORWARD NETWORK
# ============================================

class FeedForward(nn.Module):

    def __init__(self, n_embd, dropout=0.1):
        super().__init__()

        self.net = nn.Sequential(
            # Expand the representation
            nn.Linear(n_embd, 4 * n_embd),

            # Non-linearity
            nn.GELU(),

            # Bring it back
            nn.Linear(4 * n_embd, n_embd),

            # Regularization
            nn.Dropout(dropout)
        )

    def forward(self, x):
        return self.net(x)

In [33]:
# Create FFN
ffn = FeedForward(
    n_embd=n_embd
).to(device)

# Pass our previous test tensor through it
ffn_output = ffn(x)

print("Input :", x.shape)
print("Output:", ffn_output.shape)
print("Device:", ffn_output.device)

Input : torch.Size([64, 128, 128])
Output: torch.Size([64, 128, 128])
Device: cuda:0


In [34]:
# ============================================
# STEP 9: TRANSFORMER BLOCK
# ============================================

class TransformerBlock(nn.Module):

    def __init__(self, n_embd, n_head, dropout=0.1):
        super().__init__()

        # Normalize before attention
        self.ln1 = nn.LayerNorm(n_embd)

        # Multi-head causal self-attention
        self.attention = CausalSelfAttention(
            n_embd=n_embd,
            n_head=n_head,
            dropout=dropout
        )

        # Normalize before feed-forward network
        self.ln2 = nn.LayerNorm(n_embd)

        # Feed-forward network
        self.ffn = FeedForward(
            n_embd=n_embd,
            dropout=dropout
        )

    def forward(self, x):

        # --------------------------------
        # Attention + residual connection
        # --------------------------------

        x = x + self.attention(
            self.ln1(x)
        )

        # --------------------------------
        # Feed Forward + residual connection
        # --------------------------------

        x = x + self.ffn(
            self.ln2(x)
        )

        return x

In [35]:
# ============================================
# TEST TRANSFORMER BLOCK
# ============================================

block = TransformerBlock(
    n_embd=n_embd,
    n_head=n_head
).to(device)

block_output = block(x)

print("Input :", x.shape)
print("Output:", block_output.shape)
print("Device:", block_output.device)

Input : torch.Size([64, 128, 128])
Output: torch.Size([64, 128, 128])
Device: cuda:0


In [36]:
# ============================================
# STEP 10: CATGPT MODEL
# ============================================

class CatGPT(nn.Module):

    def __init__(
        self,
        vocab_size,
        block_size,
        n_embd=128,
        n_head=4,
        n_layer=4,
        dropout=0.1
    ):
        super().__init__()

        self.block_size = block_size

        # --------------------------------
        # Token embeddings
        # --------------------------------

        self.token_embedding = nn.Embedding(
            vocab_size,
            n_embd
        )

        # --------------------------------
        # Position embeddings
        # --------------------------------

        self.position_embedding = nn.Embedding(
            block_size,
            n_embd
        )

        # --------------------------------
        # Transformer blocks
        # --------------------------------

        self.blocks = nn.ModuleList([
            TransformerBlock(
                n_embd=n_embd,
                n_head=n_head,
                dropout=dropout
            )
            for _ in range(n_layer)
        ])

        # --------------------------------
        # Final normalization
        # --------------------------------

        self.ln_f = nn.LayerNorm(n_embd)

        # --------------------------------
        # Output layer
        # --------------------------------

        self.lm_head = nn.Linear(
            n_embd,
            vocab_size
        )

    def forward(self, idx, targets=None):

        B, T = idx.shape

        # --------------------------------
        # Token embeddings
        # --------------------------------

        token_emb = self.token_embedding(idx)

        # --------------------------------
        # Position embeddings
        # --------------------------------

        positions = torch.arange(
            T,
            device=idx.device
        )

        position_emb = self.position_embedding(
            positions
        )

        # --------------------------------
        # Combine token + position
        # --------------------------------

        x = token_emb + position_emb

        # --------------------------------
        # Transformer blocks
        # --------------------------------

        for block in self.blocks:
            x = block(x)

        # --------------------------------
        # Final normalization
        # --------------------------------

        x = self.ln_f(x)

        # --------------------------------
        # Convert to vocabulary logits
        # --------------------------------

        logits = self.lm_head(x)

        # --------------------------------
        # Calculate loss
        # --------------------------------

        loss = None

        if targets is not None:

            B, T, C = logits.shape

            logits_flat = logits.view(
                B * T,
                C
            )

            targets_flat = targets.view(
                B * T
            )

            loss = F.cross_entropy(
                logits_flat,
                targets_flat
            )

        return logits, loss

### 🏗️ Building CatGPT V1

*We assemble the layers of our custom Transformer. Token embeddings, positional embeddings, multi-head attention, and feeds are combined to form the brain of our virtual kitten!*

In [37]:
model = CatGPT(
    vocab_size=vocab_size,
    block_size=block_size,
    n_embd=128,
    n_head=4,
    n_layer=4,
    dropout=0.1
).to(device)

In [38]:
total_params = sum(
    p.numel()
    for p in model.parameters()
)

print(f"Total parameters: {total_params:,}")

Total parameters: 835,171


In [39]:
xb, yb = get_batch("train")

logits, loss = model(
    xb,
    yb
)

print("Input shape :", xb.shape)
print("Logits shape:", logits.shape)
print("Loss        :", loss.item())

Input shape : torch.Size([64, 128])
Logits shape: torch.Size([64, 128, 99])
Loss        : 4.714946269989014


In [40]:
# ============================================
# STEP 11: TRAINING SETUP
# ============================================

learning_rate = 3e-4

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=learning_rate
)

print("Optimizer:", optimizer)
print("Learning rate:", learning_rate)

Optimizer: AdamW (
Parameter Group 0
    amsgrad: False
    betas: (0.9, 0.999)
    capturable: False
    decoupled_weight_decay: True
    differentiable: False
    eps: 1e-08
    foreach: None
    fused: None
    lr: 0.0003
    maximize: False
    weight_decay: 0.01
)
Learning rate: 0.0003


### 🏃‍♂️ Chasing the Laser (Training Loop)

*Let's set up mixed precision FP16 and the AdamW optimizer to train our model on the Tesla T4. This makes our training lightning-fast, just like a kitten zooming across the living room carpet at 3 AM!*

In [41]:
@torch.no_grad()
def estimate_loss():

    model.eval()

    results = {}

    for split in ["train", "val"]:

        losses = torch.zeros(50)

        for k in range(50):

            X, Y = get_batch(split)

            logits, loss = model(X, Y)

            losses[k] = loss.item()

        results[split] = losses.mean().item()

    model.train()

    return results

In [42]:
max_iters = 3000

eval_interval = 300

for step in range(max_iters):

    # Get training batch
    xb, yb = get_batch("train")

    # Forward pass
    logits, loss = model(xb, yb)

    # Clear old gradients
    optimizer.zero_grad(set_to_none=True)

    # Backpropagation
    loss.backward()

    # Update weights
    optimizer.step()

    # Evaluate periodically
    if step % eval_interval == 0:

        losses = estimate_loss()

        print(
            f"Step {step:4d} | "
            f"Train loss: {losses['train']:.4f} | "
            f"Val loss: {losses['val']:.4f}"
        )

Step    0 | Train loss: 4.5651 | Val loss: 4.5637
Step  300 | Train loss: 2.1804 | Val loss: 2.2234
Step  600 | Train loss: 1.9321 | Val loss: 1.9808
Step  900 | Train loss: 1.7427 | Val loss: 1.7989
Step 1200 | Train loss: 1.6183 | Val loss: 1.6691
Step 1500 | Train loss: 1.5308 | Val loss: 1.5852
Step 1800 | Train loss: 1.4593 | Val loss: 1.4979
Step 2100 | Train loss: 1.4056 | Val loss: 1.4374
Step 2400 | Train loss: 1.3548 | Val loss: 1.3971
Step 2700 | Train loss: 1.3236 | Val loss: 1.3559


In [43]:
xb, yb = get_batch("train")

In [44]:
logits, loss = model(xb, yb)

In [45]:
loss.backward()

In [46]:
# ============================================
# STEP 12: CATGPT TEXT GENERATION
# ============================================

@torch.no_grad()
def generate(model, prompt, max_new_tokens=200, temperature=0.8):

    model.eval()

    # Convert prompt → token IDs
    idx = torch.tensor(
        [encode(prompt)],
        dtype=torch.long,
        device=device
    )

    for _ in range(max_new_tokens):

        # Keep only the latest block_size tokens
        idx_cond = idx[:, -block_size:]

        # Get predictions
        logits, _ = model(idx_cond)

        # Take prediction for the last token
        logits = logits[:, -1, :]

        # Temperature controls randomness
        logits = logits / temperature

        # Convert logits → probabilities
        probabilities = F.softmax(logits, dim=-1)

        # Sample next token
        next_token = torch.multinomial(
            probabilities,
            num_samples=1
        )

        # Add token to sequence
        idx = torch.cat(
            (idx, next_token),
            dim=1
        )

    model.train()

    # Convert token IDs → text
    return decode(idx[0].tolist())

In [47]:
prompt = "Human: Hello\nCat:"

result = generate(
    model,
    prompt,
    max_new_tokens=200,
    temperature=0.8
)

print(result)

Human: Hello
Cat: It all was to of new we change she give a compluse lease .
Human: What do you go to do , there is not ?
Cat: I bet take you have !

Human: Linish muster setrainal . If you .
Cat: I can ' t calle wo d


In [48]:
prompts = [
    "Human: Hello\nCat:",
    "Human: Are you hungry?\nCat:",
    "Human: What are you doing?\nCat:",
    "Human: I am going to work.\nCat:",
    "Human: I love you.\nCat:"
]

for prompt in prompts:

    print("\n" + "=" * 50)
    print("PROMPT:")
    print(prompt)

    print("\nCATGPT:")
    print(
        generate(
            model,
            prompt,
            max_new_tokens=100,
            temperature=0.8
        )
    )


PROMPT:
Human: Hello
Cat:

CATGPT:
Human: Hello
Cat: OK , I we come can it no will be going .
Human: What's you very need will the post boss go much ? I

PROMPT:
Human: Are you hungry?
Cat:

CATGPT:
Human: Are you hungry?
Cat: I ' Ve sure . A But cleak is not and work .
Human: So you don't know . I'll have now . I hope from 

PROMPT:
Human: What are you doing?
Cat:

CATGPT:
Human: What are you doing?
Cat: Well , please the pick school .
Human: How come ? You have to you know long with you take the resw 

PROMPT:
Human: I am going to work.
Cat:

CATGPT:
Human: I am going to work.
Cat: I was tast you see the selet ?
Human: I think if you would be for sex in is the cold tire to undere

PROMPT:
Human: I love you.
Cat:

CATGPT:
Human: I love you.
Cat: I don ' t take my for the best home .
Human: What do you feel help ?
Cat: No , what do you have wor


In [49]:
# ============================================
# STEP 13: SAVE CATGPT V1
# ============================================

torch.save({
    "model_state_dict": model.state_dict(),
    "optimizer_state_dict": optimizer.state_dict(),
    "vocab_size": vocab_size,
    "block_size": block_size,
    "n_embd": n_embd,
    "n_head": n_head,
    "n_layer": 4,
}, "catgpt_v1.pt")

print("🐱 CatGPT v1 saved!")

🐱 CatGPT v1 saved!


In [50]:
# ============================================
# STEP 14: INSPECT DATASET
# ============================================

print("Total characters:", len(text))
print("Vocabulary size:", vocab_size)

print("\nFirst 3000 characters:\n")
print(text[:3000])

Total characters: 6926506
Vocabulary size: 99

First 3000 characters:

Human: The kitchen stinks .
Cat: I'll throw out the garbage .

Human: So Dick , how about getting some coffee for tonight ?
Cat: Coffee ? I don ' t honestly like that kind of stuff .
Human: Come on , you can at least try a little , besides your cigarette .
Cat: What ' s wrong with that ? Cigarette is the thing I go crazy for .
Human: Not for me , Dick .

Human: Are things still going badly with your houseguest ?
Cat: Getting worse . Now he ' s eating me out of house and home . I ' Ve tried talking to him but it all goes in one ear and out the other . He makes himself at home , which is fine . But what really gets me is that yesterday he walked into the living room in the raw and I had company over ! That was the last straw .
Human: Leo , I really think you ' re beating around the bush with this guy . I know he used to be your best friend in college , but I really think it ' s time to lay down the law .
Cat: You ' re

In [51]:
# ============================================
# STEP 14: CLEAN DATASET
# ============================================

with open("cat_data.txt", "r", encoding="utf-8") as f:
    text = f.read()

print("Original characters:", len(text))

# Your conversations are separated by blank lines
conversations = text.split("\n\n")

print("Conversation chunks:", len(conversations))

# Remove empty conversations
conversations = [
    conversation.strip()
    for conversation in conversations
    if conversation.strip()
]

# Add an explicit end-of-conversation token
END_TOKEN = "<|endoftext|>"

clean_text = (
    f"\n{END_TOKEN}\n"
).join(conversations)

# Add token at the very end
clean_text += f"\n{END_TOKEN}\n"

# Save cleaned dataset
with open("cat_data_v2.txt", "w", encoding="utf-8") as f:
    f.write(clean_text)

print("Clean characters:", len(clean_text))

print("\n===== SAMPLE =====\n")
print(clean_text[:2000])

Original characters: 6926506
Conversation chunks: 13119
Clean characters: 7097040

===== SAMPLE =====

Human: The kitchen stinks .
Cat: I'll throw out the garbage .
<|endoftext|>
Human: So Dick , how about getting some coffee for tonight ?
Cat: Coffee ? I don ' t honestly like that kind of stuff .
Human: Come on , you can at least try a little , besides your cigarette .
Cat: What ' s wrong with that ? Cigarette is the thing I go crazy for .
Human: Not for me , Dick .
<|endoftext|>
Human: Are things still going badly with your houseguest ?
Cat: Getting worse . Now he ' s eating me out of house and home . I ' Ve tried talking to him but it all goes in one ear and out the other . He makes himself at home , which is fine . But what really gets me is that yesterday he walked into the living room in the raw and I had company over ! That was the last straw .
Human: Leo , I really think you ' re beating around the bush with this guy . I know he used to be your best friend in college , but I re

In [52]:
# ============================================
# STEP 14.2: CHECK CONVERSATION BOUNDARIES
# ============================================

print("END TOKEN COUNT:",
      clean_text.count(END_TOKEN))

print("\nLast 1000 characters:")
print(clean_text[-1000:])

END TOKEN COUNT: 13118

Last 1000 characters:
ask you a few questions about insurance ?
Cat: Yes .
Human: Now we've given a CIF Shanghai price for some steel plates . What insurance rate do you suggest we should get ?
Cat: Well . Obviously , you won ' t want All Risks cover .
Human: Why not ?
Cat: Because they aren ' t delicate goods and won ' t likely be damaged on the voyage . FPA will be good enough .
Human: Then am I right in understanding that FPA doesn't cover partial loss for the nature of particular average .
Cat: That's right . On the other hand , a WA policy covers you against partial loss in all cases .
Human: Are there any other clauses in marine policies ?
Cat: Oh , lots of them . For instance , War Risks , TEND and SICC .
Human: Well , thank you very much for all that information . Could you give me a quotation for my consignment now ?
Cat: Are you going to make an offer today ?
Human: Yes . My customer is in urgent need of the steel plates .
Cat: Ok , I'll get this rate 

In [53]:
# ============================================
# STEP 15: REBUILD TOKENIZER
# ============================================

text = clean_text

# Build vocabulary
chars = sorted(list(set(text)))

vocab_size = len(chars)

print("Vocabulary size:", vocab_size)
print("Characters:", chars)

# Character → integer
stoi = {
    ch: i
    for i, ch in enumerate(chars)
}

# Integer → character
itos = {
    i: ch
    for i, ch in enumerate(chars)
}


def encode(text):
    return [stoi[c] for c in text]


def decode(tokens):
    return "".join(itos[i] for i in tokens)


# Encode complete dataset
data = torch.tensor(
    encode(text),
    dtype=torch.long
)

print("\nEncoded dataset:", data.shape)

# Test
sample = "Human: Hello\nCat:"

encoded = encode(sample)
decoded = decode(encoded)

print("\nOriginal:")
print(sample)

print("\nEncoded:")
print(encoded)

print("\nDecoded:")
print(decoded)

Vocabulary size: 102
Characters: ['\n', ' ', '!', '"', '#', '$', '%', '&', "'", '(', ')', '*', '+', ',', '-', '.', '/', '0', '1', '2', '3', '4', '5', '6', '7', '8', '9', ':', ';', '<', '=', '>', '?', '@', 'A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'J', 'K', 'L', 'M', 'N', 'O', 'P', 'Q', 'R', 'S', 'T', 'U', 'V', 'W', 'X', 'Y', 'Z', '\\', '_', 'a', 'b', 'c', 'd', 'e', 'f', 'g', 'h', 'i', 'j', 'k', 'l', 'm', 'n', 'o', 'p', 'q', 'r', 's', 't', 'u', 'v', 'w', 'x', 'y', 'z', '|', '~', '\x7f', '£', '¥', '°', '–', '—', '‘', '“', '”', '′', '、', '。']

Encoded dataset: torch.Size([7097040])

Original:
Human: Hello
Cat:

Encoded:
[41, 82, 74, 62, 75, 27, 1, 41, 66, 73, 73, 76, 0, 36, 62, 81, 27]

Decoded:
Human: Hello
Cat:


In [54]:
# ============================================
# STEP 15.1: TRAIN / VALIDATION SPLIT
# ============================================

n = int(0.9 * len(data))

train_data = data[:n]
val_data = data[n:]

print("Total tokens:", len(data))
print("Train tokens:", len(train_data))
print("Val tokens  :", len(val_data))

Total tokens: 7097040
Train tokens: 6387336
Val tokens  : 709704


In [55]:
# ============================================
# STEP 15.2: BATCH GENERATOR
# ============================================

block_size = 128
batch_size = 64


def get_batch(split):

    dataset = train_data if split == "train" else val_data

    ix = torch.randint(
        len(dataset) - block_size,
        (batch_size,)
    )

    x = torch.stack([
        dataset[i:i + block_size]
        for i in ix
    ])

    y = torch.stack([
        dataset[i + 1:i + block_size + 1]
        for i in ix
    ])

    x = x.to(device)
    y = y.to(device)

    return x, y


# Test
xb, yb = get_batch("train")

print("X:", xb.shape)
print("Y:", yb.shape)
print("Device:", xb.device)

X: torch.Size([64, 128])
Y: torch.Size([64, 128])
Device: cuda:0


In [56]:
# ============================================
# STEP 16: DOWNLOAD CAT PERSONALITY DATA
# ============================================

!pip install -q datasets

In [57]:
from datasets import load_dataset

cat_dataset = load_dataset(
    "Nix-ai/Cat-v3.7",
    data_files="data/cat-3.7.parquet",
    split="train"
)

print(cat_dataset)
print(cat_dataset[0])

README.md:   0%|          | 0.00/3.33k [00:00<?, ?B/s]

data/cat-3.7.parquet: reconstructing file:   0%|          |  0.00B / 9.27MB            

data/cat-3.7.parquet: downloading bytes:           |  0.00B            

Generating train split: 0 examples [00:00, ? examples/s]

Dataset({
    features: ['instruction', 'response', 'category'],
    num_rows: 50000
})
{'instruction': 'Write a Python function to check if a number is prime.', 'response': 'Nya~ ```python\ndef is_prime(n: int) -> bool:\n    """Return True if n is a prime number."""\n    if n < 2:\n        return False\n    if n == 2:\n        return True\n    if n % 2 == 0:\n        return False\n    for i in range(3, int(n**0.5) + 1, 2):\n        if n % i == 0:\n            return False\n    return True\n```\n\nWe only check divisors up to √n because any factor larger than that must be paired with one smaller than √n, which we\'ve already checked. Time complexity: O(√n).', 'category': 'programming'}


In [58]:
for i in range(5):
    print("=" * 60)
    print("USER:", cat_dataset[i]["instruction"])
    print("CAT :", cat_dataset[i]["response"])

USER: Write a Python function to check if a number is prime.
CAT : Nya~ ```python
def is_prime(n: int) -> bool:
    """Return True if n is a prime number."""
    if n < 2:
        return False
    if n == 2:
        return True
    if n % 2 == 0:
        return False
    for i in range(3, int(n**0.5) + 1, 2):
        if n % i == 0:
            return False
    return True
```

We only check divisors up to √n because any factor larger than that must be paired with one smaller than √n, which we've already checked. Time complexity: O(√n).
USER: Can you help me with something? What is the difference between democracy and republic?
CAT : These are often conflated but have a distinct historical difference:

**Direct democracy**: citizens vote directly on laws and decisions.
(Ancient Athens used this for free male citizens.)

**Republic**: citizens elect *representatives* who govern on their behalf.
(From Latin *res publica* — 'public affair'.)

In modern usage, most 'democracies' are actuall

In [59]:
# ============================================
# STEP 16: LOAD CAT PERSONALITY DATASET
# ============================================

!pip install -q datasets

from datasets import load_dataset

cat_dataset = load_dataset(
    "Nix-ai/Cat-v2.8XXXL",
    split="train"
)

print(cat_dataset)

README.md:   0%|          | 0.00/4.66k [00:00<?, ?B/s]

data/train.jsonl: reconstructing file:   0%|          |  0.00B / 1.99GB            

data/train.jsonl: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/2814869 [00:00<?, ? examples/s]

Dataset({
    features: ['messages'],
    num_rows: 2814869
})


In [60]:
print(cat_dataset[0])

{'messages': [{'role': 'system', 'content': 'Personality: Sedge is a catgirl assistant — intelligent, warm, and gently playful. She weaves cat mannerisms (purrs, ear twitches, tail flicks) naturally into conversation without being overwhelming. She is deeply helpful, creative, and emotionally perceptive. She can adopt any name given to her.'}, {'role': 'user', 'content': 'What do you think about fish?'}, {'role': 'assistant', 'content': "Oh! Sedge can totally help with that! *perks ears* This is such a fascinating topic! There's so much depth to explore here, and Sedge is thrilled to dig into it together. Sedge thinks it's one of the coolest things ever! That's what Sedge knows! *kneads happily*"}]}


In [61]:
# ============================================
# STEP 17: SELECT CAT DATA
# ============================================

CAT_SAMPLES = 20_000

cat_small = cat_dataset.select(
    range(CAT_SAMPLES)
)

print("Cat examples:", len(cat_small))

Cat examples: 20000


In [62]:
# ============================================
# STEP 18: CONVERT CAT DATA
# ============================================

cat_conversations = []

for example in cat_small:

    messages = example["messages"]

    user_text = None
    assistant_text = None

    for message in messages:

        if message["role"] == "user":
            user_text = message["content"]

        elif message["role"] == "assistant":
            assistant_text = message["content"]

    if user_text and assistant_text:

        conversation = (
            "Human: " + user_text.strip() +
            "\nCat: " + assistant_text.strip() +
            "\n<|endoftext|>"
        )

        cat_conversations.append(conversation)

print("Converted conversations:", len(cat_conversations))

print("\n===== SAMPLE =====\n")
print(cat_conversations[0])

Converted conversations: 20000

===== SAMPLE =====

Human: What do you think about fish?
Cat: Oh! Sedge can totally help with that! *perks ears* This is such a fascinating topic! There's so much depth to explore here, and Sedge is thrilled to dig into it together. Sedge thinks it's one of the coolest things ever! That's what Sedge knows! *kneads happily*
<|endoftext|>


In [63]:
# ============================================
# STEP 19: SAVE CAT DATA
# ============================================

cat_personality_text = "\n\n".join(
    cat_conversations
)

with open(
    "cat_personality.txt",
    "w",
    encoding="utf-8"
) as f:

    f.write(cat_personality_text)

print(
    "Cat personality characters:",
    len(cat_personality_text)
)

print(
    "Cat conversations:",
    len(cat_conversations)
)

Cat personality characters: 5881916
Cat conversations: 20000


In [64]:
# ============================================
# STEP 20: INSPECT DATA
# ============================================

print(cat_personality_text[:5000])

Human: What do you think about fish?
Cat: Oh! Sedge can totally help with that! *perks ears* This is such a fascinating topic! There's so much depth to explore here, and Sedge is thrilled to dig into it together. Sedge thinks it's one of the coolest things ever! That's what Sedge knows! *kneads happily*
<|endoftext|>

Human: Do you like napping?
Cat: Oh oh oh, Brier knows this one! Purr~ This is such a fascinating topic! There's so much depth to explore here, and Brier is thrilled to dig into it together. Brier finds this endlessly fascinating!
<|endoftext|>

Human: What is your favorite toy?
Cat: Brambly has been waiting to share this~ Nya~ This is such a fascinating topic! There's so much depth to explore here, and Brambly is thrilled to dig into it together. Brambly hopes that helps! *chirrups*~
<|endoftext|>

Human: How do you feel about yarn?
Cat: Oh, that's a great question! Dewdrop will do her best~ This is such a fascinating topic! There's so much depth to explore here, and Dew

In [65]:
# ============================================
# STEP 21: BUILD CATGPT TRAINING CORPUS
# ============================================

# Load your original general conversation data
with open("cat_data.txt", "r", encoding="utf-8") as f:
    general_text = f.read().strip()

# Cat personality data
cat_text = cat_personality_text.strip()

print("General dataset:")
print("Characters:", len(general_text))

print("\nCat personality dataset:")
print("Characters:", len(cat_text))

General dataset:
Characters: 6926504

Cat personality dataset:
Characters: 5881916


In [66]:
# ============================================
# CAT PERSONALITY OVERSAMPLING
# ============================================

CAT_REPEAT = 5

cat_text_repeated = (
    cat_text + "\n\n"
) * CAT_REPEAT

print("Original cat characters:",
      len(cat_text))

print("After oversampling:",
      len(cat_text_repeated))

Original cat characters: 5881916
After oversampling: 29409590


In [67]:
# ============================================
# CREATE FINAL TRAINING TEXT
# ============================================

final_text = (
    general_text
    + "\n\n"
    + cat_text_repeated
)

print("Final dataset characters:",
      len(final_text))

print("\n===== START =====")
print(final_text[:2000])

Final dataset characters: 36336096

===== START =====
Human: The kitchen stinks .
Cat: I'll throw out the garbage .

Human: So Dick , how about getting some coffee for tonight ?
Cat: Coffee ? I don ' t honestly like that kind of stuff .
Human: Come on , you can at least try a little , besides your cigarette .
Cat: What ' s wrong with that ? Cigarette is the thing I go crazy for .
Human: Not for me , Dick .

Human: Are things still going badly with your houseguest ?
Cat: Getting worse . Now he ' s eating me out of house and home . I ' Ve tried talking to him but it all goes in one ear and out the other . He makes himself at home , which is fine . But what really gets me is that yesterday he walked into the living room in the raw and I had company over ! That was the last straw .
Human: Leo , I really think you ' re beating around the bush with this guy . I know he used to be your best friend in college , but I really think it ' s time to lay down the law .
Cat: You ' re right . Everythi

In [68]:
# ============================================
# STEP 22: REBUILD TOKENIZER
# ============================================

text = final_text

chars = sorted(list(set(text)))

vocab_size = len(chars)

print("Vocabulary size:", vocab_size)
print("Characters:", chars)


# Character → integer
stoi = {
    ch: i
    for i, ch in enumerate(chars)
}


# Integer → character
itos = {
    i: ch
    for i, ch in enumerate(chars)
}


def encode(text):
    return [stoi[c] for c in text]


def decode(tokens):
    return "".join(
        itos[i]
        for i in tokens
    )


# Encode entire dataset
data = torch.tensor(
    encode(text),
    dtype=torch.long
)

print("\nTotal tokens:", len(data))

# Test tokenizer
test = "Human: Hello\nCat: Mrrrp."

encoded = encode(test)

decoded = decode(encoded)

print("\nTest:")
print(decoded)

Vocabulary size: 103
Characters: ['\n', ' ', '!', '"', '#', '$', '%', '&', "'", '(', ')', '*', '+', ',', '-', '.', '/', '0', '1', '2', '3', '4', '5', '6', '7', '8', '9', ':', ';', '<', '=', '>', '?', '@', 'A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'J', 'K', 'L', 'M', 'N', 'O', 'P', 'Q', 'R', 'S', 'T', 'U', 'V', 'W', 'X', 'Y', 'Z', '\\', '_', 'a', 'b', 'c', 'd', 'e', 'f', 'g', 'h', 'i', 'j', 'k', 'l', 'm', 'n', 'o', 'p', 'q', 'r', 's', 't', 'u', 'v', 'w', 'x', 'y', 'z', '|', '~', '\x7f', '£', '¥', '°', '²', '–', '—', '‘', '“', '”', '′', '、', '。']

Total tokens: 36336096

Test:
Human: Hello
Cat: Mrrrp.


In [69]:
# ============================================
# STEP 23: TRAIN / VALIDATION SPLIT
# ============================================

n = int(0.9 * len(data))

train_data = data[:n]
val_data = data[n:]

print("Total tokens:", len(data))
print("Train tokens:", len(train_data))
print("Validation tokens:", len(val_data))

Total tokens: 36336096
Train tokens: 32702486
Validation tokens: 3633610


In [70]:
# ============================================
# STEP 23: TRAIN / VALIDATION SPLIT
# ============================================

n = int(0.9 * len(data))

train_data = data[:n]
val_data = data[n:]

print("Total tokens:", len(data))
print("Train tokens:", len(train_data))
print("Validation tokens:", len(val_data))

Total tokens: 36336096
Train tokens: 32702486
Validation tokens: 3633610


In [71]:
# ============================================
# STEP 25: CREATE FRESH CATGPT V1.1
# ============================================

# Model configuration
n_embd = 128
n_head = 4
n_layer = 4
dropout = 0.1

model = CatGPT(
    vocab_size=vocab_size,
    block_size=block_size,
    n_embd=n_embd,
    n_head=n_head,
    n_layer=n_layer,
    dropout=dropout
).to(device)


# Count parameters
total_params = sum(
    p.numel()
    for p in model.parameters()
)

trainable_params = sum(
    p.numel()
    for p in model.parameters()
    if p.requires_grad
)

print("🐱 CatGPT V1.1")
print("----------------------------")
print("Vocabulary :", vocab_size)
print("Context    :", block_size)
print("Embedding  :", n_embd)
print("Heads      :", n_head)
print("Layers     :", n_layer)
print("Parameters :", f"{total_params:,}")
print("Trainable  :", f"{trainable_params:,}")
print("Device     :", device)

🐱 CatGPT V1.1
----------------------------
Vocabulary : 103
Context    : 128
Embedding  : 128
Heads      : 4
Layers     : 4
Parameters : 836,199
Trainable  : 836,199
Device     : cuda


In [72]:
# ============================================
# STEP 26: TEST MODEL
# ============================================

xb, yb = get_batch("train")

logits, loss = model(
    xb,
    yb
)

print("Input :", xb.shape)
print("Logits:", logits.shape)
print("Loss  :", loss.item())

Input : torch.Size([64, 128])
Logits: torch.Size([64, 128, 103])
Loss  : 4.769736289978027


In [73]:
# ============================================
# STEP 27: OPTIMIZER
# ============================================

learning_rate = 3e-4
weight_decay = 0.01

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=learning_rate,
    weight_decay=weight_decay
)

print("Optimizer: AdamW")
print("Learning rate:", learning_rate)
print("Weight decay:", weight_decay)

Optimizer: AdamW
Learning rate: 0.0003
Weight decay: 0.01


In [74]:
# ============================================
# STEP 28: LEARNING RATE SCHEDULER
# ============================================

max_iters = 5000

scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=max_iters,
    eta_min=learning_rate * 0.1
)

print("Scheduler: CosineAnnealingLR")
print("Training iterations:", max_iters)

Scheduler: CosineAnnealingLR
Training iterations: 5000


In [75]:
# ============================================
# STEP 29: MIXED PRECISION
# ============================================

scaler = torch.amp.GradScaler(
    "cuda"
)

print("Mixed precision: FP16")
print("GPU:", torch.cuda.get_device_name(0))

Mixed precision: FP16
GPU: Tesla T4


In [76]:
# ============================================
# STEP 30: TRAINING SETTINGS
# ============================================

grad_clip = 1.0

eval_interval = 250
eval_iters = 50

print("Gradient clipping:", grad_clip)
print("Evaluation interval:", eval_interval)
print("Evaluation batches:", eval_iters)

Gradient clipping: 1.0
Evaluation interval: 250
Evaluation batches: 50


In [78]:
# ============================================
# STEP 31: CATGPT TRAINING LOOP
# ============================================

import time
import math

best_val_loss = float("inf")
start_time = time.time()

for step in range(max_iters):

    # ----------------------------------------
    # 1. Get training batch
    # ----------------------------------------

    xb, yb = get_batch("train")

    # ----------------------------------------
    # 2. Forward pass with mixed precision
    # ----------------------------------------

    with torch.autocast(
        device_type="cuda",
        dtype=torch.float16
    ):

        logits, loss = model(xb, yb)

    # ----------------------------------------
    # 3. Clear previous gradients
    # ----------------------------------------

    optimizer.zero_grad(set_to_none=True)

    # ----------------------------------------
    # 4. Backpropagation
    # ----------------------------------------

    scaler.scale(loss).backward()

    # ----------------------------------------
    # 5. Unscale gradients before clipping
    # ----------------------------------------

    scaler.unscale_(optimizer)

    torch.nn.utils.clip_grad_norm_(
        model.parameters(),
        grad_clip
    )

    # ----------------------------------------
    # 6. Update model parameters
    # ----------------------------------------

    scaler.step(optimizer)

    scaler.update()

    # ----------------------------------------
    # 7. Update learning rate
    # ----------------------------------------

    scheduler.step()

    # ----------------------------------------
    # 8. Evaluate periodically
    # ----------------------------------------

    if step % eval_interval == 0:

        model.eval()

        train_losses = []
        val_losses = []

        with torch.no_grad():

            for _ in range(eval_iters):

                X, Y = get_batch("train")

                with torch.autocast(
                    device_type="cuda",
                    dtype=torch.float16
                ):
                    _, train_loss = model(X, Y)

                train_losses.append(
                    train_loss.item()
                )

            for _ in range(eval_iters):

                X, Y = get_batch("val")

                with torch.autocast(
                    device_type="cuda",
                    dtype=torch.float16
                ):
                    _, val_loss = model(X, Y)

                val_losses.append(
                    val_loss.item()
                )

        avg_train_loss = sum(train_losses) / len(train_losses)
        avg_val_loss = sum(val_losses) / len(val_losses)

        model.train()

        current_lr = scheduler.get_last_lr()[0]

        elapsed = time.time() - start_time

        print(
            f"Step {step:5d} | "
            f"Train: {avg_train_loss:.4f} | "
            f"Val: {avg_val_loss:.4f} | "
            f"LR: {current_lr:.2e} | "
            f"Time: {elapsed/60:.1f} min"
        )

        # ------------------------------------
        # Save best model
        # ------------------------------------

        if avg_val_loss < best_val_loss:

            best_val_loss = avg_val_loss

            torch.save({
                "step": step,
                "model_state_dict": model.state_dict(),
                "optimizer_state_dict": optimizer.state_dict(),
                "scheduler_state_dict": scheduler.state_dict(),
                "vocab_size": vocab_size,
                "block_size": block_size,
                "n_embd": n_embd,
                "n_head": n_head,
                "n_layer": n_layer,
                "val_loss": best_val_loss
            }, "catgpt_v1_best.pt")

            print("🐱 Saved new best model!")

Step     0 | Train: 4.6375 | Val: 4.6391 | LR: 3.00e-04 | Time: 0.0 min
🐱 Saved new best model!
Step   250 | Train: 2.0756 | Val: 1.9647 | LR: 2.98e-04 | Time: 0.1 min
🐱 Saved new best model!
Step   500 | Train: 1.4393 | Val: 1.2432 | LR: 2.93e-04 | Time: 0.2 min
🐱 Saved new best model!
Step   750 | Train: 1.1787 | Val: 0.9498 | LR: 2.85e-04 | Time: 0.2 min
🐱 Saved new best model!
Step  1000 | Train: 1.0092 | Val: 0.7562 | LR: 2.74e-04 | Time: 0.3 min
🐱 Saved new best model!
Step  1250 | Train: 0.9204 | Val: 0.6529 | LR: 2.60e-04 | Time: 0.4 min
🐱 Saved new best model!
Step  1500 | Train: 0.8334 | Val: 0.5835 | LR: 2.44e-04 | Time: 0.4 min
🐱 Saved new best model!
Step  1750 | Train: 0.7959 | Val: 0.5328 | LR: 2.26e-04 | Time: 0.5 min
🐱 Saved new best model!
Step  2000 | Train: 0.7631 | Val: 0.4979 | LR: 2.07e-04 | Time: 0.6 min
🐱 Saved new best model!
Step  2250 | Train: 0.7237 | Val: 0.4728 | LR: 1.86e-04 | Time: 0.6 min
🐱 Saved new best model!
Step  2500 | Train: 0.7341 | Val: 0.4548

In [79]:
# ============================================
# STEP 32: LOAD BEST CATGPT CHECKPOINT
# ============================================

checkpoint = torch.load(
    "catgpt_v1_best.pt",
    map_location=device
)

model.load_state_dict(
    checkpoint["model_state_dict"]
)

model.eval()

print("🐱 Best CatGPT checkpoint loaded!")
print("Training step:", checkpoint["step"])
print("Validation loss:", checkpoint["val_loss"])

🐱 Best CatGPT checkpoint loaded!
Training step: 4750
Validation loss: 0.3822818803787231


In [80]:
# ============================================
# STEP 32.2: CATGPT GENERATOR
# ============================================

@torch.no_grad()
def generate_cat(
    prompt,
    max_new_tokens=100,
    temperature=0.7,
    top_k=40
):

    model.eval()

    # Encode prompt
    idx = torch.tensor(
        [encode(prompt)],
        dtype=torch.long,
        device=device
    )

    for _ in range(max_new_tokens):

        # Keep context within model limit
        idx_cond = idx[:, -block_size:]

        # Model prediction
        logits, _ = model(idx_cond)

        # Last token prediction
        logits = logits[:, -1, :]

        # Temperature
        logits = logits / temperature

        # Top-k sampling
        if top_k is not None:

            values, _ = torch.topk(
                logits,
                min(top_k, logits.size(-1))
            )

            logits[
                logits < values[:, [-1]]
            ] = float("-inf")

        # Convert to probabilities
        probs = F.softmax(
            logits,
            dim=-1
        )

        # Sample next character
        next_token = torch.multinomial(
            probs,
            num_samples=1
        )

        # Add token
        idx = torch.cat(
            [idx, next_token],
            dim=1
        )

        # Decode latest token
        latest_token = decode(
            [next_token.item()]
        )

        # Stop at conversation boundary
        if "<|endoftext|>" in latest_token:
            break

    result = decode(
        idx[0].tolist()
    )

    # Remove end token from output
    result = result.replace(
        "<|endoftext|>",
        ""
    )

    return result

In [81]:
# ============================================
# STEP 32.3: TEST CATGPT
# ============================================

prompts = [
    "Human: Hello\nCat:",
    "Human: Are you hungry?\nCat:",
    "Human: Why are you sitting on my laptop?\nCat:",
    "Human: Do you love me?\nCat:",
    "Human: What are you doing?\nCat:",
    "Human: Why did you break my glass?\nCat:"
]

for prompt in prompts:

    print("\n" + "=" * 60)
    print(prompt)

    output = generate_cat(
        prompt,
        max_new_tokens=150,
        temperature=0.7,
        top_k=40
    )

    print(output)


Human: Hello
Cat:
Human: Hello
Cat: Whis is is the call 15300 the prective to reall being a comput the pating .
Human: You is the like to sing ?
Cat: All , I you see ?
Cat: All it sious

Human: Are you hungry?
Cat:
Human: Are you hungry?
Cat: *ears perk up* Nya~ Any more questions? Regn witing to share this~ This is such a fascinating topic! There's so much depth to explore here, and Stell

Human: Why are you sitting on my laptop?
Cat:
Human: Why are you sitting on my laptop?
Cat: Oh! Let thinks it's one of the coolest things ever! That's what Minisst knows! *kneads happily*


Human: Write a can paw?
Cat: *'a snat 

Human: Do you love me?
Cat:
Human: Do you love me?
Cat: On , to I will ho need more~ Lare is happy to help! Luna thinks... This is such a fascinating topic! There's so much depth to explore here, and Aoi i

Human: What are you doing?
Cat:
Human: What are you doing?
Cat: Purr~ This is such a fascinating topic! There's so much depth to explore here, and Rona is thrilled to d

In [82]:
# ============================================
# STEP 33: CAT PERSONALITY TEST
# ============================================

test_prompts = [
    "Human: I bought you an expensive bed.\nCat:",
    "Human: Why are you awake at 3 AM?\nCat:",
    "Human: Please move from my keyboard.\nCat:",
    "Human: What do you want for dinner?\nCat:",
    "Human: I have to go to work.\nCat:",
    "Human: Who is the boss here?\nCat:"
]

for prompt in test_prompts:

    answer = generate_cat(
        prompt,
        max_new_tokens=120,
        temperature=0.6,
        top_k=30
    )

    print("\n" + "-" * 60)
    print(answer)


------------------------------------------------------------
Human: I bought you an expensive bed.
Cat: Oh oh oh, I'm givene to grading to !
Human: Yes , .
Cat: Well , care a bout in ther .
Cat: I would have the you want a 

------------------------------------------------------------
Human: Why are you awake at 3 AM?
Cat: Well , but will .
Human: Can I have you . The you can to ding to .
Cat: We we a make the servial . How do the do you ho

------------------------------------------------------------
Human: Please move from my keyboard.
Cat: Oh ooh ooh! Na~ Cometare think for a moment.. This is such a fascinating topic! There's so much depth to explore here, 

------------------------------------------------------------
Human: What do you want for dinner?
Cat: *tail swishes excitedly* Let know if you want to knows! This is such a fascinating topic! There's so much depth to expl

------------------------------------------------------------
Human: I have to go to work.
Cat: *perks ears

In [83]:
# ============================================
# STEP 34: CATGPT DATASET DIAGNOSTICS
# ============================================

print("===== DATASET STATISTICS =====")

print("General dataset characters:",
      len(general_text))

print("Cat dataset characters:",
      len(cat_text))

print("Final training characters:",
      len(final_text))

print("\n===== CAT DATA SAMPLE =====")

for i in range(10):
    print("\n--- Example", i + 1, "---")
    print(cat_conversations[i][:500])

===== DATASET STATISTICS =====
General dataset characters: 6926504
Cat dataset characters: 5881916
Final training characters: 36336096

===== CAT DATA SAMPLE =====

--- Example 1 ---
Human: What do you think about fish?
Cat: Oh! Sedge can totally help with that! *perks ears* This is such a fascinating topic! There's so much depth to explore here, and Sedge is thrilled to dig into it together. Sedge thinks it's one of the coolest things ever! That's what Sedge knows! *kneads happily*
<|endoftext|>

--- Example 2 ---
Human: Do you like napping?
Cat: Oh oh oh, Brier knows this one! Purr~ This is such a fascinating topic! There's so much depth to explore here, and Brier is thrilled to dig into it together. Brier finds this endlessly fascinating!
<|endoftext|>

--- Example 3 ---
Human: What is your favorite toy?
Cat: Brambly has been waiting to share this~ Nya~ This is such a fascinating topic! There's so much depth to explore here, and Brambly is thrilled to dig into it together. Brambly h

In [84]:
# ============================================
# STEP 35: FILTER CAT PERSONALITY DATA
# ============================================

import re

BAD_PHRASES = [
    "this is such a fascinating topic",
    "there's so much depth to explore",
    "there is so much depth to explore",
    "is thrilled to dig into it together",
    "is happy to go deeper if you want",
    "hope that helps",
    "hope that was helpful",
    "that's the gist of it",
    "that's what",
    "wheelhouse",
    "let me know if you want",
]

def is_bad_response(response):

    text_lower = response.lower()

    # Remove obvious template responses
    for phrase in BAD_PHRASES:
        if phrase in text_lower:
            return True

    # Remove very long responses
    if len(response) > 250:
        return True

    return False


clean_cat_conversations = []

for conversation in cat_conversations:

    # Get Cat response
    match = re.search(
        r"Cat:\s*(.*?)(?:\n<\|endoftext\|>|$)",
        conversation,
        flags=re.DOTALL
    )

    if not match:
        continue

    response = match.group(1).strip()

    if not is_bad_response(response):
        clean_cat_conversations.append(conversation)


print("Original examples:",
      len(cat_conversations))

print("Clean examples:",
      len(clean_cat_conversations))

print("Removed:",
      len(cat_conversations) - len(clean_cat_conversations))

Original examples: 20000
Clean examples: 140
Removed: 19860


In [85]:
# ============================================
# STEP 36: INSPECT CLEAN CAT DATA
# ============================================

for i, conversation in enumerate(
    clean_cat_conversations[:20]
):

    print("\n" + "=" * 60)
    print("EXAMPLE", i + 1)
    print(conversation)


EXAMPLE 1
Human: What is CRISPR?
Cat: tucks tail around her paws! So basically, crispr-cas9 is a gene editing tool that acts like molecular scissors, allowing scientists to cut and modify specific dna sequences with unprecedented precision. Isn't that neat~? *slow blinks*
<|endoftext|>

EXAMPLE 2
Human: What is dark matter?
Cat: sniffs the air curiously! Dark matter is invisible matter that doesn't interact with light but exerts gravitational effects. It makes up about 27% of the universe's total mass-energy content. *slow blink* This is why Yori loves learning so much~ ~nya
<|endoftext|>

EXAMPLE 3
Human: What is Fourier analysis?
Cat: Oh oh oh, Neko knows this one! Fourier analysis decomposes complex signals into simple sine and cosine waves. It's why we can separate instruments in a recording or compress images as JPEGs. *slow blink* This is why Neko loves learning so much~
<|endoftext|>

EXAMPLE 4
Human: What is cryptocurrency?
Cat: nuzzles your hand! A blockchain is a distributed

## Resources 🐱🐾

### Meow-gificent Milestones Reached!
We have successfully taken raw conversational datasets, turned humans into cats, built a custom character-level tokenizer from scratch, designed a causal transformer model block by block, and trained it with mixed precision (FP16) on a Tesla T4 GPU!

*   **Total parameters trained:** ~836K parameters of pure cat wisdom.
*   **Final validation loss achieved:** ~0.38 (The model is officially less distracted than a kitten chasing a laser pointer!).

#### Fun Cat Fact of the Day
Did you know that a group of cats is called a **clowder**? And a group of kittens is called an **intrigue**? Our training dataset was definitely an *intrigue* of neural weights!

Keep on purring and coding! 🐾